In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from utils import data_loader_utils
DATA_ROOT = Path("./data")
TARGET_PROCESS_NUM = 7
MACHINES = {"M01", "M02", "M03"}   # only these machines
SAVE_PREFIX = "raw_stats_OP07"     # output filenames


# -----------------------
# Helpers
# -----------------------
def parse_process_number(op_name: str):
    m = re.search(r"(\d+)", op_name)
    return int(m.group(1)) if m else None

def load_xyz(file_path: Path) -> np.ndarray:
    arr = data_loader_utils.datafile_read(file_path, False)
    xyz = np.asarray(arr[:, :3], dtype=float)  # x,y,z
    # keep only fully finite rows (raw, no normalization)
    xyz = xyz[np.isfinite(xyz).all(axis=1)]
    return xyz

def stats_xyz(xyz: np.ndarray):

    n = xyz.shape[0]
    if n == 0:
        mean = np.array([np.nan, np.nan, np.nan])
        var  = np.array([np.nan, np.nan, np.nan])
        rms  = np.array([np.nan, np.nan, np.nan])
        return n, mean, var, rms

    mean = xyz.mean(axis=0)
    var  = xyz.var(axis=0, ddof=0)                 # population variance
    rms  = np.sqrt((xyz * xyz).mean(axis=0))       # sqrt(mean(x^2))
    return n, mean, var, rms


def iter_op7_files(data_root: Path):

    for machine_dir in sorted([p for p in data_root.iterdir() if p.is_dir()]):
        machine = machine_dir.name
        if machine not in MACHINES:
            continue

        for op_dir in sorted([p for p in machine_dir.iterdir() if p.is_dir()]):
            proc_num = parse_process_number(op_dir.name)
            if proc_num != TARGET_PROCESS_NUM:
                continue

            for label in ("good", "bad"):
                lab_dir = op_dir / label
                if not lab_dir.exists():
                    continue
                for fp in sorted(lab_dir.glob("*.h5")):
                    yield machine, op_dir.name, label, fp


# -----------------------
# Main: per-file + per-(machine,label) for OP7
# -----------------------
rows = []

for machine, op_name, label, fp in iter_op7_files(DATA_ROOT):
    xyz = load_xyz(fp)
    n, mean, var, rms = stats_xyz(xyz)

    rows.append({
        "machine": machine,
        "op_folder": op_name,
        "label": label,
        "file": fp.name,
        "n_samples": n,
        "mean_x": mean[0], "mean_y": mean[1], "mean_z": mean[2],
        "var_x":  var[0],  "var_y":  var[1],  "var_z":  var[2],
        "rms_x":  rms[0],  "rms_y":  rms[1],  "rms_z":  rms[2],
    })

df_files = pd.DataFrame(rows).sort_values(["machine", "label", "file"]).reset_index(drop=True)

# Aggregate across all OP7 files for each machine+label
def agg_stats(group: pd.DataFrame):
    machine = group["machine"].iloc[0]
    label = group["label"].iloc[0]
    op_folder = group["op_folder"].iloc[0]

    all_xyz = []
    for fname in group["file"].tolist():
        fp = DATA_ROOT / machine / op_folder / label / fname
        all_xyz.append(load_xyz(fp))
    xyz = np.concatenate(all_xyz, axis=0) if all_xyz else np.empty((0, 3))

    n, mean, var, rms = stats_xyz(xyz)
    return pd.Series({
        "n_samples": n,
        "mean_x": mean[0], "mean_y": mean[1], "mean_z": mean[2],
        "var_x":  var[0],  "var_y":  var[1],  "var_z":  var[2],
        "rms_x":  rms[0],  "rms_y":  rms[1],  "rms_z":  rms[2],
    })

if len(df_files) == 0:
    print("No OP7 files found with the expected folder structure for M01/M02/M03.")
    df_groups = pd.DataFrame()
else:
    df_groups = (
        df_files.groupby(["machine", "op_folder", "label"], as_index=False)
        .apply(agg_stats)
        .reset_index(drop=True)
        .sort_values(["machine", "label"])
        .reset_index(drop=True)
    )

# Save
files_out  = f"{SAVE_PREFIX}_by_file.csv"
groups_out = f"{SAVE_PREFIX}_by_machine_label.csv"

df_files.to_csv(files_out, index=False)
df_groups.to_csv(groups_out, index=False)

print("Saved:")
print(" -", files_out)
print(" -", groups_out)

display(df_groups)

Saved:
 - raw_stats_OP07_by_file.csv
 - raw_stats_OP07_by_machine_label.csv


C:\Users\Younes\AppData\Local\Temp\ipykernel_12004\3426842744.py:132: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(agg_stats)


,machine,op_folder,label,n_samples,mean_x,mean_y,mean_z,var_x,var_y,var_z,rms_x,rms_y,rms_z
0,M01,OP07,bad,157710.0,-0.975379,31.242286,-1023.803335,281149.003572,100075.415909,115372.890383,530.235754,317.885980,1078.677968
1,M01,OP07,good,2151456.0,-3.076094,33.549539,-1017.237887,169872.763379,34504.427539,19550.617543,412.167716,188.759103,1026.802579
2,M02,OP07,bad,109649.0,16.252770,12.498564,-1043.916059,250849.302285,113792.078907,103323.239368,501.112218,337.562280,1092.283836
3,M02,OP07,good,2612720.0,2.278439,4.491877,-1032.956165,163850.584535,23086.944515,24124.529398,404.791027,152.010268,1044.568317
4,M03,OP07,bad,105472.0,5.893877,12.715849,-1021.230971,275707.442479,92354.406307,67075.837495,525.111588,304.164592,1053.559934
5,M03,OP07,good,2563112.0,11.233587,17.275546,-1021.339695,204696.263140,46676.884043,48867.516026,452.573151,216.737926,1044.989134


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from statsmodels.tsa.stattools import adfuller, kpss

from utils import data_loader_utils
from utils import user_defined_functions as udf

# -----------------------
# CONFIG
# -----------------------
DATA_ROOT = Path("./data")
OP_FOLDER = "OP07"
MACHINES = ["M01", "M02", "M03"]
LABELS = ["good", "bad"] 
AXES = ["X", "Y", "Z"]

ORIGINAL_FS = 2000
TARGET_FS = 1000

WS = 4096
STEP = WS // 2
ALPHA = 0.05

MAX_WINDOWS_PER_GROUP = None  # e.g. 500


# -----------------------
# LOAD / RESAMPLE / WINDOW
# -----------------------
def list_h5_files(machine: str, op: str, label: str):
    folder = DATA_ROOT / machine / op / label
    return sorted(folder.glob("*.h5"))

def load_xyz(fp: Path) -> np.ndarray:
    arr = data_loader_utils.datafile_read(fp, False)
    xyz = np.asarray(arr[:, :3], dtype=float)     # x,y,z
    xyz = xyz[np.isfinite(xyz).all(axis=1)]
    return xyz

def resample_xyz(xyz: np.ndarray, fs_in=ORIGINAL_FS, fs_out=TARGET_FS) -> np.ndarray:
    xs = udf.resample_milling_data(xyz[:, 0], fs_in, fs_out)
    ys = udf.resample_milling_data(xyz[:, 1], fs_in, fs_out)
    zs = udf.resample_milling_data(xyz[:, 2], fs_in, fs_out)
    out = np.stack([xs, ys, zs], axis=1)
    out = out[np.isfinite(out).all(axis=1)]
    return out

def sliding_windows(xyz: np.ndarray, ws=WS, step=STEP):
    n = xyz.shape[0]
    for start in range(0, n - ws + 1, step):
        yield start, xyz[start:start + ws]


# -----------------------
# STATIONARITY TESTS (per 1D window)
# -----------------------
def test_stationarity_window(x: np.ndarray, alpha=ALPHA):
    """
    ADF:   H0 = unit root (non-stationary). want p < alpha to support stationarity.
    KPSS:  H0 = stationary.                want p > alpha to support stationarity.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    # guardrails
    if x.size < 50 or np.allclose(x, x[0]):
        return dict(adf_p=np.nan, kpss_p=np.nan, adf_stationary=False, kpss_stationary=False, both_stationary=False, ok=False)

    try:
        adf_p = adfuller(x, autolag="AIC", regression="c")[1]
    except Exception:
        adf_p = np.nan

    try:
        kpss_p = kpss(x, regression="c", nlags="auto")[1]
    except Exception:
        kpss_p = np.nan

    ok = np.isfinite(adf_p) and np.isfinite(kpss_p)
    adf_stat = ok and (adf_p < alpha)
    kpss_stat = ok and (kpss_p > alpha)
    both = adf_stat and kpss_stat

    return dict(adf_p=adf_p, kpss_p=kpss_p, adf_stationary=adf_stat, kpss_stationary=kpss_stat, both_stationary=both, ok=ok)


# -----------------------
# RUN
# -----------------------
rows = []

for mach in MACHINES:
    for lab in LABELS:
        files = list_h5_files(mach, OP_FOLDER, lab)
        if not files:
            print(f"[WARN] No files found for {mach}/{OP_FOLDER}/{lab}")
            continue

        tested = 0
        for fp in files:
            xyz = resample_xyz(load_xyz(fp))  # resampled raw
            for w_idx, (start, w) in enumerate(sliding_windows(xyz)):
                if MAX_WINDOWS_PER_GROUP is not None and tested >= MAX_WINDOWS_PER_GROUP:
                    break
                tested += 1

                for ai, ax in enumerate(AXES):
                    out = test_stationarity_window(w[:, ai], alpha=ALPHA)
                    rows.append({
                        "machine": mach,
                        "label": lab,
                        "axis": ax,
                        "file": fp.name,
                        "window_start": int(start),
                        "adf_p": out["adf_p"],
                        "kpss_p": out["kpss_p"],
                        "adf_stationary": out["adf_stationary"],
                        "kpss_stationary": out["kpss_stationary"],
                        "both_stationary": out["both_stationary"],
                        "ok": out["ok"],
                    })

            if MAX_WINDOWS_PER_GROUP is not None and tested >= MAX_WINDOWS_PER_GROUP:
                break

df_windows = pd.DataFrame(rows)
if df_windows.empty:
    raise RuntimeError("No windows tested. Check DATA_ROOT / OP_FOLDER / folder structure.")

# -----------------------
# SUMMARY per machine+label+axis
# -----------------------
df_ok = df_windows[df_windows["ok"]].copy()

summary = (
    df_ok.groupby(["machine", "label", "axis"], as_index=False)
    .agg(
        n_windows=("both_stationary", "size"),
        adf_stationary_rate=("adf_stationary", "mean"),
        kpss_stationary_rate=("kpss_stationary", "mean"),
        both_stationary_rate=("both_stationary", "mean"),
        median_adf_p=("adf_p", "median"),
        median_kpss_p=("kpss_p", "median"),
    )
    .sort_values(["machine", "label", "axis"])
    .reset_index(drop=True)
)

# Pivot view: stationary rate per axis (ADF+KPSS agreement)
pivot = summary.pivot_table(
    index=["machine", "label"],
    columns="axis",
    values="both_stationary_rate",
    aggfunc="first"
).reset_index()

print("=== Stationarity results for OP07 (windowed, resampled raw) ===")
display(summary)
display(pivot)

# Save if you want
df_windows.to_csv("stationarity_OP07_windows.csv", index=False)
summary.to_csv("stationarity_OP07_summary.csv", index=False)
pivot.to_csv("stationarity_OP07_pivot.csv", index=False)

print("Saved: stationarity_OP07_windows.csv, stationarity_OP07_summary.csv, stationarity_OP07_pivot.csv")


C:\Users\Younes\AppData\Local\Temp\ipykernel_22044\837366706.py:87: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = kpss(x, regression="c", nlags="auto")[1]
C:\Users\Younes\AppData\Local\Temp\ipykernel_22044\837366706.py:87: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = kpss(x, regression="c", nlags="auto")[1]
C:\Users\Younes\AppData\Local\Temp\ipykernel_22044\837366706.py:87: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_p = kpss(x, regression="c", nlags="auto")[1]
C:\Users\Younes\AppData\Local\Temp\ipykernel_22044\837366706.py:87: InterpolationWarning: The test statistic is outside of the range of p-v